# Reto Vientos

Dos pasos separados:

| Paso | Qué hace | Red |
|---|---|---|
| **1 · Crawling** | Descarga con crawl4ai el viento de todas las estaciones | sí |
| **2 · Datos** | Revisa las lecturas en tablas, las guarda en `retos/output/vientos/<fecha>/` y comprueba que los ficheros están completos | no |

| Fuente | Qué es | Clave |
|---|---|---|
| AEMET OpenData | ~855 estaciones automáticas (observación) | AEMET |
| Puertos del Estado (iMar / PORTUS) | ~100 boyas, mareógrafos y estaciones de puerto (observación) | no |
| Open-Meteo | 52 capitales de provincia (modelo, respaldo) | no |

Cada lectura incluye:
- velocidad y racha (km/h);
- de dónde viene (`direccion_grados`) y hacia dónde empuja (`hacia_grados`);
- temperatura y humedad;
- `regla_30` (>30 °C, <30 % de humedad y >30 km/h);
- `antiguedad_min`: minutos desde la medida. No se descarta ninguna estación por ir atrasada.

Solo produce datos, no abre ningún mapa: para verlos en el mapa interactivo está el pipeline completo (`pipeline_incendios.ipynb`).

## 0. Preparación

In [ ]:
# Recarga automática: si el código de incendios/ o retos/ cambia, se usa la versión nueva sin reiniciar.
%load_ext autoreload
%autoreload 2

import os
import sys
from pathlib import Path

# Solo si el notebook no encuentra el proyecto (la carpeta que contiene `incendios/`), escribe aquí su ruta:
RUTA_PROYECTO = ""
if RUTA_PROYECTO:
    os.environ["INCENDIOS_RAIZ"] = RUTA_PROYECTO

# El reto vive junto a este notebook; al importarlo localiza el proyecto solo (ver `_localizar_proyecto`).
if str(Path.cwd()) not in sys.path:
    sys.path.insert(0, str(Path.cwd()))
import reto_vientos
RAIZ = reto_vientos.RAIZ

import pandas as pd

from incendios.config import fijar_claves, estado_claves
from incendios.pipeline import resumen
from reto_vientos import comprobar, crawlear, guardar, leer_json, nueva_carpeta

pd.set_option("display.max_columns", 30)
pd.set_option("display.width", 200)
print("Proyecto:", RAIZ)

## 1. Claves

- **AEMET:** necesaria para las ~855 estaciones de AEMET.
- **Sin ella:** Puertos del Estado y Open-Meteo funcionan igual.
- **LSA-SAF:** no se usa en este reto.

Si ya están en `.env` (por ejemplo, porque las guardaste desde el pipeline), deja las comillas vacías. ⚠️ Si las pegas aquí, no subas este notebook a git.

In [ ]:
AEMET_API_KEY   = ""
LSASAF_USUARIO  = ""   # opcional (SEVIRI)
LSASAF_PASSWORD = ""   # opcional (SEVIRI)

fijar_claves(aemet=AEMET_API_KEY, lsasaf_usuario=LSASAF_USUARIO, lsasaf_password=LSASAF_PASSWORD)
for variable, valor in estado_claves().items():
    print(f"{variable:<16} {valor}")

# PASO 1 · Crawling

Las tres fuentes se lanzan en paralelo con crawl4ai. Cada una va aislada: si falla o le falta la clave, queda anotada y las demás siguen.

### 1.1 Ejecutar el crawling

In [ ]:
resultado = await crawlear()
print(resumen(resultado["estado"]))
df = pd.DataFrame(resultado["lecturas"])
display(df.groupby(["fuente", "tipo_dato"]).size().rename("estaciones").to_frame())

# PASO 2 · Datos

Revisión de las lecturas, guardado en ficheros (`viento.geojson`, `estado.json` y `manifiesto.json`) y comprobación de que se han escrito completos. No hace peticiones de red: trabaja con lo que ha descargado el paso 1.

### 2.1 Rachas más fuertes

In [ ]:
columnas = ["fuente", "nombre", "antiguedad_min", "velocidad_kmh", "racha_kmh", "direccion_cardinal", "temperatura_c", "humedad_pct", "regla_30"]
display(df.sort_values("racha_kmh", ascending=False)[columnas].head(15))

### 2.2 Condiciones de riesgo

Estaciones con la humedad más baja y las que cumplen la regla del 30, el umbral de condiciones extremas de propagación.

In [ ]:
print("Estaciones en regla del 30:", int(df["regla_30"].fillna(False).sum()))
display(df.sort_values("humedad_pct")[columnas].head(10))

### 2.3 Antigüedad de las lecturas

No se descarta ninguna estación por ir atrasada; aquí se ve cuántas tienen una lectura reciente.

In [ ]:
tramos = pd.cut(df["antiguedad_min"], [-1, 30, 90, 180, 360, float("inf")],
                labels=["≤ 30 min", "30–90 min", "1,5–3 h", "3–6 h", "> 6 h"])
display(tramos.value_counts().sort_index().rename("estaciones").to_frame())

### 2.4 Guardar los ficheros

Escribe `viento.geojson`, `estado.json` y `manifiesto.json` en una carpeta nueva. El manifiesto indica qué fichero es cada capa y cuántos registros tiene, para que cualquier otra aplicación pueda leerlos.

In [ ]:
carpeta = nueva_carpeta("vientos")
guardar(resultado, carpeta)
print("Guardado en", carpeta)
manifiesto = leer_json(carpeta / "manifiesto.json")
print(manifiesto["capas"]["viento"]["por_fuente"])

### 2.5 Comprobar los ficheros guardados

Se relee lo que se acaba de escribir y se compara con el manifiesto y con lo que hay en memoria: cada fichero tiene que existir, poder leerse y tener todos sus registros.
- **Todo ✓:** los datos están completos.
- **Algún ✗:** vuelve a ejecutar la sección anterior.

In [ ]:
tabla = pd.DataFrame(comprobar(carpeta))
tabla["ok"] = tabla["ok"].map({True: "✓", False: "✗"})
display(tabla)
print("Datos completos ✓" if (tabla["ok"] == "✓").all() else "Hay ficheros incompletos ✗: vuelve a guardar")
print()
print("Carpeta de datos:", carpeta)
for f in sorted(carpeta.rglob("*")):
    if f.is_file():
        print(f"   {f.relative_to(carpeta)}  ({f.stat().st_size // 1024} KB)")

### Mapa interactivo

Genera `mapa_vientos.html` en la carpeta de datos y lo abre en el navegador. Lee solo los ficheros guardados: no hace peticiones.

In [ ]:
from reto_vientos import mapa
ruta = mapa(carpeta, abrir=True)
print("Mapa:", ruta)